# 01 - Ingesta Bronze
Este notebook implementa la ingesta batch del dataset de operaciones BYMA hacia la capa bronze.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbfs_path = "dbfs:/Volumes/workspace/default/iol_data/operaciones_byma_2026.csv"

In [0]:
schema = StructType([
    StructField("fecha", StringType(), True),
    StructField("tipoTran", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("descripcion_titulo", StringType(), True),
    StructField("moneda", StringType(), True),
    StructField("simbolo_titulo", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DecimalType(18, 4), True),
    StructField("id_transaccion", StringType(), True),
    StructField("origen", StringType(), True)
])

In [0]:
df_raw = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv(dbfs_path)

In [0]:
df = df_raw \
    .withColumn("fecha_ts_utc", to_timestamp("fecha")) \
    .withColumn("fecha_ts_local", from_utc_timestamp("fecha_ts_utc", "America/Argentina/Buenos_Aires")) \
    .withColumn("fecha_particion", to_date("fecha_ts_local")) \
    .withColumn("_ingested_at", current_timestamp())

In [0]:
#Duplicados
df_duplicates = df.groupBy("id_transaccion") \
    .count() \
    .withColumnRenamed("count", "count_dup") \
    .filter(col("count_dup") > 1) \
    .withColumn("fecha_deteccion", current_timestamp())

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_byma")
spark.catalog.tableExists("bronze_byma.calidad_duplicados")

In [0]:
df_duplicates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_byma.calidad_duplicados")

In [0]:

fechas = [row["fecha_particion"] for row in df.select("fecha_particion").distinct().collect()]
fechas = sorted(fechas)

In [0]:
table_name = "bronze_byma.operaciones_raw"

for fecha in fechas:
    
    df_batch_day = df.filter(col("fecha_particion") == fecha)
    
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        
        delta_table.alias("target").merge(
            df_batch_day.alias("source"),
            "target.id_transaccion = source.id_transaccion"
        ).whenNotMatchedInsertAll().execute()
        
    else:
        df_batch_day.write \
            .format("delta") \
            .partitionBy("fecha_particion") \
            .saveAsTable(table_name)

# Se utiliza delta para lograr idempotente (re-ejecutarlo no genera duplicados)

Validaciones

In [0]:
# Conteo vs input (perdida de datos)
df.count()

In [0]:
%sql
SELECT COUNT(*) FROM bronze_byma.operaciones_raw

In [0]:
%sql 
-- Clientes únicos (~57.6k)
SELECT COUNT(DISTINCT id_cliente) 
FROM bronze_byma.operaciones_raw

In [0]:
%sql
-- Duplicados en tabla final
SELECT id_transaccion, COUNT(*) 
FROM bronze_byma.operaciones_raw
GROUP BY id_transaccion
HAVING COUNT(*) > 1

In [0]:
%sql
-- Control de particiones (YYYY-MM-DD)
SHOW PARTITIONS bronze_byma.operaciones_raw

In [0]:
# Fechas
display(df.select("fecha", "fecha_ts_utc", "fecha_ts_local").limit(10))

In [0]:
%sql
-- Precios (no negativos, ni absurdos)
SELECT MIN(precio), MAX(precio)
FROM bronze_byma.operaciones_raw

In [0]:
%sql
-- Distintas monedas
SELECT moneda, COUNT(*)
FROM bronze_byma.operaciones_raw
GROUP BY moneda

In [0]:
%sql
-- Distribución transacciones
SELECT tipoTran, COUNT(*)
FROM bronze_byma.operaciones_raw
GROUP BY tipoTran

In [0]:
%sql
-- Nulos en ID
SELECT COUNT(*) 
FROM bronze_byma.operaciones_raw
WHERE id_transaccion IS NULL OR id_cliente IS NULL

In [0]:
%sql
-- Canales: 5
SELECT origen, COUNT(*) 
FROM bronze_byma.operaciones_raw
GROUP BY origen

In [0]:
%sql
-- Rango Ene–Mar 2026
SELECT 
    MIN(fecha_ts_local) AS min_fecha,
    MAX(fecha_ts_local) AS max_fecha
FROM bronze_byma.operaciones_raw

In [0]:
%sql
SELECT 
    id_transaccion,
    COUNT(*) AS cantidad
FROM bronze_byma.operaciones_raw
GROUP BY id_transaccion
HAVING COUNT(*) > 1
ORDER BY cantidad DESC;

In [0]:
%sql
SELECT *
FROM bronze_byma.operaciones_raw
LIMIT 25
--WHERE id_cliente = 'CLIDBBA0462'